In [1]:
import pandas as pd
import concurrent.futures
import langid
from deep_translator import GoogleTranslator
from googletrans import Translator
import time

In [2]:
counter = 0

In [3]:
def get_translated_combined_description(appid, df):
    """
    Translates the description for games in chunks of 4900 characters, because GoogleTranslator
    does not accept text that has >5000 characters.

    return: the translated game description
    """

    global counter
    counter += 1

    print(f"Translating game no. {counter} with appid: {appid}")

    description = df.loc[df['appid'] == appid, 'combined_description'].iloc[0]

    description_chunks = []

    for i in range(0, len(description), 200):
        description_chunks.append(description[i:i + 200])

    translated_chunks = []
    for chunk in description_chunks:
        try:
            translated_chunk = GoogleTranslator(source='auto', target='en').translate(chunk)
        except Exception as e:
            print(f"Error {e} for the appid: {appid}")
            translated_chunk = ""

        if translated_chunk is None:
            translated_chunk = ""  # or fallback to original chunk

        translated_chunks.append(translated_chunk)

        time.sleep(1)

    translated_description = ""

    translated_description += (str(appid) + " ")

    for translated_chunk in translated_chunks:
        translated_description += translated_chunk

    return translated_description

In [1]:
def get_foreign_combined_descriptions_appids(df):
    """
    Get all the appids for games with foreign language in the description
    :return: all the appids for games with foreign language in the description
    """
    foreign_combined_descriptions_appids = []

    for i in range(len(df)):
        description = df.iloc[i]['combined_description']

        if pd.isna(description):
            continue

        lang, _ = langid.classify(df.iloc[i]['combined_description'])

        if lang != 'en':
            foreign_combined_descriptions_appids.append(int(df.iloc[i]['appid']))

    print("Appids for games with foreign description: ", foreign_combined_descriptions_appids)
    return foreign_combined_descriptions_appids

In [5]:
# Translating the combined_description and saving the translated data in a new csv

df = pd.read_csv('../../data/7_steam_games_cleaned_data.csv')

translated_combined_descriptions = []

foreign_combined_descriptions_appids = get_foreign_combined_descriptions_appids(df)

for appid in foreign_combined_descriptions_appids:
    translated_description = get_translated_combined_description(appid, df)
    translated_combined_descriptions.append(translated_description)

for translated_combined_description in translated_combined_descriptions:
    appid = translated_combined_description.split()[0]
    translated_description = translated_combined_description.split(" ", 1)[1]

    df.loc[df['appid'] == int(appid), 'combined_description'] = translated_description

df.to_csv('../../data/8_steam_games_translated_data.csv', index=False)

Appids for games with foreign description:  [291480, 526870, 394360, 264710, 1238840, 905370, 848450, 602960, 493520, 1447430, 1062090, 383080, 2456740, 1189490, 489630, 1108320, 1437880, 570780, 2909400, 447850, 893180, 1880330, 2361390, 2059790, 1084520, 892420, 570770, 1028470, 1461810, 2656540, 1437820, 825380, 1466060, 1127700, 1606180, 1348300, 1993150, 921390, 970500, 1242700, 2668510, 1764390, 1412510, 1437840, 1410090, 3157850, 1437830, 1499520, 3028310, 616090, 913550, 1058770, 1126310, 2478970, 1863750, 1424720, 2106520, 1132730, 2340650, 1546570, 1414210, 1109460, 2011470, 2153950, 528060, 2400340, 1310990, 2458530, 716710, 1167790, 2237720, 2182400, 880950, 1225780, 836640, 1347970, 2875140, 1393350, 2757350, 1115640, 1538460, 1461990, 1434480, 2446270, 1536070, 1939560, 907950, 787450, 2139310, 2059810, 590910, 1524160, 1057680, 2850190, 1164100, 2479720, 1209000, 2795530, 934580, 314860, 1693990, 2930120, 412600, 1250650, 2664960, 1186460, 1672670, 353540, 1266630, 13457

In [5]:
# There has been some errors when translating the data, so in this cell I will translate the data that has not been translated and save in a new .csv file

df = pd.read_csv("../../data/8_steam_games_translated_data.csv")

translated_combined_descriptions = []

foreign_combined_descriptions_appids = get_foreign_combined_descriptions_appids(df)

df['combined_description'] = df['combined_description'].fillna(".") # Fill the empty values with an empty string

for appid in foreign_combined_descriptions_appids:
    translated_description = get_translated_combined_description(appid, df)
    translated_combined_descriptions.append(translated_description)

for translated_combined_description in translated_combined_descriptions:
    appid = translated_combined_description.split()[0]
    translated_description = translated_combined_description.split(" ", 1)[1]

    df.loc[df['appid'] == int(appid), 'combined_description'] = translated_description

df.to_csv('../../data/9_steam_games_all_translated_data.csv', index=False)

Appids for games with foreign description:  [291480, 1764390, 1410090, 528060, 1250650, 353540, 1482380, 719750, 1657150, 2206660, 508950, 1149400, 650510, 2341920, 2304030, 2409570, 635250, 1093880, 2314720, 2546710, 3402530, 1920770, 1477450, 2258590, 1508650, 1665780, 498620, 1010870, 765650, 2446190, 3254460, 570230, 2490930, 949970, 2983140, 2112520, 774721, 1128200, 2836600, 2010900, 925470, 1486330, 2467400, 1058150, 1929850, 1291140, 3476540, 1730980, 1259390, 1621680, 1110690, 1052370, 2027920, 710190, 2105010, 698950, 1600900, 1553230, 1967450, 2261090, 455640, 1642580, 1120370, 1879640, 1064440, 819790, 951130, 2502420, 2400290, 1864440, 2303320, 636310, 1471450, 1569930, 1313940, 1191770, 836080, 2463800, 1095120, 2566880, 1632310, 602150, 491520, 1528370, 1114950, 1591650, 831660, 1494920, 2283940, 2548950, 1071070, 461830, 1182830, 1589890, 2718560, 1145130, 1512710, 1489120, 2056990, 2629340, 1893020, 1829260, 2606820, 1707450, 987480, 1545290, 678470, 2549700, 3405620, 

In [6]:
df = pd.read_csv("../../data/9_steam_games_all_translated_data.csv")

df['combined_description'] = df['combined_description'].fillna("")

foreign_combined_descriptions_appids = get_foreign_combined_descriptions_appids(df)

Appids for games with foreign description:  [291480, 1764390, 1410090, 528060, 1250650, 353540, 1482380, 719750, 1657150, 2206660, 508950, 1149400, 2341920, 2304030, 635250, 2314720, 2546710, 1920770, 2258590, 1508650, 1665780, 1010870, 765650, 2446190, 3254460, 570230, 2490930, 949970, 2112520, 774721, 1128200, 1486330, 2467400, 1929850, 1291140, 1259390, 1621680, 1110690, 1052370, 2027920, 2105010, 698950, 1553230, 2261090, 1642580, 1120370, 1879640, 1064440, 819790, 951130, 2400290, 1864440, 2303320, 1471450, 1569930, 1313940, 1191770, 836080, 2463800, 1095120, 2566880, 1632310, 1591650, 1494920, 2548950, 1071070, 1182830, 1589890, 2718560, 1145130, 1512710, 1489120, 2056990, 2629340, 1893020, 1829260, 2606820, 1707450, 987480, 1545290, 2549700, 1110720, 1598590, 1367960, 2285650, 1480540, 1357960, 1666090, 2447790, 1276300, 772460, 3404170, 3222630, 579160, 1483600, 1645880, 2985170, 698880, 719470, 1572460, 1950200, 1289120, 2053950, 2645280, 1545810, 925120, 2081830, 2085390, 186